In [2]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 beautifulsoup4 transformers torch sentence-transformers faiss-cpu accelerate spacy networkx pyvis bitsandbytes -q
!python -m spacy download en_core_web_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 92.6 MB/s eta 0:00:00
✔ Download and installation successful
You can 

In [3]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Setup Persistent Directories
APP_DIR = "/content/drive/MyDrive/PolicyNav"
os.makedirs(APP_DIR, exist_ok=True)
os.makedirs(os.path.join(APP_DIR, 'documents'), exist_ok=True)
os.makedirs(os.path.join(APP_DIR, 'graphs'), exist_ok=True)
os.environ['APP_DIR'] = APP_DIR

print(f"✅ Persistent App Directory set to: {APP_DIR}")
print(f"👉 Please upload your policy PDFs/files directly to {os.path.join(APP_DIR, 'documents')} in your Google Drive.")



Mounted at /content/drive
✅ Persistent App Directory set to: /content/drive/MyDrive/PolicyNav
👉 Please upload your policy PDFs/files directly to /content/drive/MyDrive/PolicyNav/documents in your Google Drive.


In [4]:
%%writefile config.py
import os

JWT_SECRET = os.getenv("JWT_SECRET_KEY")
JWT_ALGORITHM = "HS256"
TOKEN_EXPIRY_MINUTES = 60

MAX_LOGIN_ATTEMPTS = 3
LOCK_TIME_MINUTES = 5
PASSWORD_HISTORY_COUNT = 3

EMAIL_ID = os.getenv("EMAIL_ID")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

ADMIN_EMAIL = os.getenv("ADMIN_EMAIL_ID")
ADMIN_PASSWORD = os.getenv("ADMIN_PASSWORD")

Writing config.py


In [5]:
%%writefile db.py
import os
import sqlite3
import json
from datetime import datetime

DB_NAME = os.path.join(os.environ.get('APP_DIR', '.'), "policynav_users.db")

def get_connection():
    return sqlite3.connect(DB_NAME, check_same_thread=False)

def init_db():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE,
        email TEXT UNIQUE,
        password_hash TEXT,
        security_question TEXT,
        security_answer_hash TEXT,
        failed_attempts INTEGER DEFAULT 0,
        lock_until TEXT,
        password_history TEXT
    )
    """)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS activity_log (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_email TEXT,
        app_section TEXT,
        user_input TEXT,
        ai_output TEXT,
        created_at TEXT DEFAULT CURRENT_TIMESTAMP
    )
    """)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS feedback (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_email TEXT,
        section TEXT,
        rating INTEGER,
        comments TEXT,
        created_at TEXT DEFAULT CURRENT_TIMESTAMP
    )
    """)

    conn.commit()
    conn.close()

def create_user(username, email, password_hash, question, answer_hash):
    conn = get_connection()
    cursor = conn.cursor()
    try:
        history = json.dumps([password_hash])
        cursor.execute("""
        INSERT INTO users (username, email, password_hash, security_question,
                           security_answer_hash, password_history)
        VALUES (?, ?, ?, ?, ?, ?)
        """, (username, email, password_hash, question, answer_hash, history))
        conn.commit()
        return True, "User created"
    except:
        return False, "User exists"
    finally:
        conn.close()

def get_user_by_email(email):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users WHERE email=?", (email,))
    user = cursor.fetchone()
    conn.close()
    return user

def update_login_attempts(email, attempts, lock_until=None):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE users SET failed_attempts=?, lock_until=? WHERE email=?",
        (attempts, lock_until, email)
    )
    conn.commit()
    conn.close()

def update_password(email, new_hash):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE users SET password_hash=? WHERE email=?", (new_hash, email))
    conn.commit()
    conn.close()

def update_password_history(email, history_json):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE users SET password_history=? WHERE email=?",
        (history_json, email)
    )
    conn.commit()
    conn.close()

def get_all_users():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT username, email, failed_attempts, lock_until FROM users")
    rows = cursor.fetchall()
    conn.close()
    return rows

# ---------- FEEDBACK ----------
def submit_feedback(user_email, section, rating, comments):

    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO feedback (user_email, section, rating, comments)
        VALUES (?, ?, ?, ?)
    """, (user_email, section, rating, comments))

    conn.commit()
    conn.close()

def log_activity(user_email, section, user_input, ai_output):

    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO activity_log (user_email, app_section, user_input, ai_output)
        VALUES (?, ?, ?, ?)
    """, (user_email, section, user_input, ai_output))

    conn.commit()
    conn.close()

def get_user_activity(user_email):

    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT app_section, user_input, ai_output, created_at
        FROM activity_log
        WHERE user_email=?
        ORDER BY created_at DESC
    """, (user_email,))

    rows = cursor.fetchall()

    conn.close()
    return rows

def get_all_feedback():

    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT user_email, section, rating, comments, created_at
        FROM feedback
        ORDER BY created_at DESC
    """)

    rows = cursor.fetchall()

    conn.close()
    return rows

Writing db.py


In [6]:
%%writefile auth.py
import re
import bcrypt
import jwt
import random
import smtplib
from datetime import datetime, timedelta
from email.mime.text import MIMEText
from config import *

# ---------- HASHING ----------
def hash_text(text: str) -> str:
    return bcrypt.hashpw(text.encode(), bcrypt.gensalt()).decode()

def verify_text(text: str, hashed: str) -> bool:
    return bcrypt.checkpw(text.encode(), hashed.encode())

# ---------- EMAIL VALIDATION ----------
def validate_email(email: str) -> bool:
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return re.match(pattern, email) is not None

# ---------- PASSWORD VALIDATION ----------
def validate_password(password: str):
    if len(password) < 8:
        return False, "Password must be at least 8 characters"
    if not re.search(r"[A-Z]", password):
        return False, "Password must contain at least 1 uppercase letter"
    if not re.search(r"[a-z]", password):
        return False, "Password must contain at least 1 lowercase letter"
    if not re.search(r"\d", password):
        return False, "Password must contain at least 1 number"
    return True, "Valid password"

# ---------- JWT ----------
def generate_token(email: str):
    payload = {
        "email": email,
        "exp": datetime.utcnow() + timedelta(minutes=TOKEN_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm=JWT_ALGORITHM)

def verify_token(token: str):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=[JWT_ALGORITHM])
    except:
        return None

# ---------- OTP ----------
def generate_otp():
    return str(random.randint(100000, 999999))

def send_otp_email(receiver, otp):
    html_content = f"""
    <html>
    <body style="font-family: Arial, sans-serif; background-color: #f4f4f4; padding: 20px;">
        <div style="
            max-width: 400px;
            margin: auto;
            background: white;
            padding: 20px;
            border-radius: 10px;
            text-align: center;
            box-shadow: 0 4px 10px rgba(0,0,0,0.1);
        ">
            <h2 style="color: #2c3e50;">PolicyNav Verification</h2>
            <p style="color: #555;">Use the OTP below to continue:</p>

            <div style="
                font-size: 28px;
                font-weight: bold;
                letter-spacing: 3px;
                color: #00b894;
                margin: 20px 0;
            ">
                {otp}
            </div>

            <p style="color: #999; font-size: 12px;">
                This OTP is valid for 10 minutes.
            </p>
        </div>
    </body>
    </html>
    """

    msg = MIMEText(html_content, "html")
    msg["Subject"] = "PolicyNav OTP Verification"
    msg["From"] = EMAIL_ID
    msg["To"] = receiver

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL_ID, EMAIL_APP_PASSWORD)
        server.send_message(msg)

Writing auth.py


In [7]:
%%writefile readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text

    def get_all_metrics(self):

        metrics = {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text),
            "Coleman-Liau": textstat.coleman_liau_index(self.text)
        }

        # ⭐ Overall Grade Level (average of 4 grade metrics)
        metrics["Overall Grade Level"] = (
            metrics["Flesch-Kincaid Grade"]
            + metrics["SMOG Index"]
            + metrics["Gunning Fog"]
            + metrics["Coleman-Liau"]
        ) / 4

        # ⭐ Raw text statistics
        metrics["Total Sentences"] = textstat.sentence_count(self.text)
        metrics["Total Words"] = textstat.lexicon_count(self.text)
        metrics["Total Syllables"] = textstat.syllable_count(self.text)
        metrics["Complex Words"] = textstat.difficult_words(self.text)
        metrics["Total Characters"] = len(self.text)

        return metrics

Writing readability.py


In [8]:
%%writefile vector_store.py
import os, glob, pickle, faiss, PyPDF2
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer

APP_DIR = os.environ.get('APP_DIR', '.')
INDEX_PATH = os.path.join(APP_DIR, "faiss_index.bin")
META_PATH = os.path.join(APP_DIR, "faiss_meta.pkl")

# We only load it once cached in Streamlit
_embedder = None

def get_embedder():
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    return _embedder

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        for page in PyPDF2.PdfReader(pdf_path).pages: text += page.extract_text() + "\n"
    except: pass
    return text

def ingest_documents(docs_dir):
    if not os.path.exists(docs_dir): return 0

    # Load existing metdata to avoid re-ingesting
    existing_metadata = []
    if os.path.exists(META_PATH):
        try:
            with open(META_PATH, 'rb') as f: existing_metadata = pickle.load(f)
        except: pass

    existing_filenames = set([d['filename'] for d in existing_metadata])

    files = glob.glob(os.path.join(docs_dir, "*"))
    new_chunks = []
    new_metadata = []

    for filepath in files:
        filename = os.path.basename(filepath)
        if filename in existing_filenames: continue # Skip already ingested!

        print(f"Parsing new document: {filename}")
        text = ""
        if filepath.lower().endswith(".pdf"): text = extract_text_from_pdf(filepath)
        elif filepath.lower().endswith((".htm", ".html")): text = BeautifulSoup(open(filepath, 'r').read(), 'html.parser').get_text(separator=' ')
        elif filepath.lower().endswith(".txt"): text = open(filepath, 'r').read()

        if text.strip():
            for i in range(0, len(text), 1500):
                chunk = text[i:i+1500]
                if len(chunk) > 50:
                    new_chunks.append(chunk)
                    new_metadata.append({"filename": filename, "content": chunk})

    if not new_chunks: return 0 # Nothing new to ingest

    embedder = get_embedder()
    embeddings = embedder.encode(new_chunks, convert_to_numpy=True)

    if os.path.exists(INDEX_PATH):
        index = faiss.read_index(INDEX_PATH)
    else:
        index = faiss.IndexFlatL2(embeddings.shape[1])

    index.add(embeddings)
    faiss.write_index(index, INDEX_PATH)

    final_metadata = existing_metadata + new_metadata
    with open(META_PATH, 'wb') as f: pickle.dump(final_metadata, f)
    return len(new_chunks)

def search_documents(query, top_k=5):
    if not os.path.exists(INDEX_PATH) or not os.path.exists(META_PATH): return []
    try:
        embedder = get_embedder()
        index = faiss.read_index(INDEX_PATH)
        with open(META_PATH, 'rb') as f: metadata = pickle.load(f)
        distances, indices = index.search(embedder.encode([query], convert_to_numpy=True), top_k)

        # Deduplicate to try to get broader file sources rather than just 5 chunks from the exact same page
        results = []
        seen_files = set()
        for idx in indices[0]:
            if idx != -1 and idx < len(metadata):
                doc = metadata[idx]
                fname = doc['filename']
                if list(seen_files).count(fname) < 2: # Max 2 chunks from same file allowed
                    results.append(doc)
                    seen_files.add(fname)
        return results
    except: return []

def get_all_documents():
    if not os.path.exists(META_PATH): return []
    with open(META_PATH, 'rb') as f: return pickle.load(f)

Writing vector_store.py


In [9]:
%%writefile knowledge_graph.py
import spacy, networkx as nx, os
from pyvis.network import Network

try: nlp = spacy.load("en_core_web_sm")
except: nlp = None

APP_DIR = os.environ.get('APP_DIR', '.')

def build_graph_from_documents(docs):
    G = nx.Graph()
    max_docs = min(len(docs), 50)
    if not nlp: return G

    for i in range(max_docs):
        text = docs[i]['content'][:1000]
        doc = nlp(text)
        entities = [
            (ent.text.strip(), ent.label_) for ent in doc.ents
            if ent.label_ in ['ORG','GPE','LAW','PERSON'] and len(ent.text.strip()) > 2
        ]

        entities = list(dict.fromkeys(entities))[:15]

        source_node = f"Doc: {docs[i]['filename']}"
        G.add_node(
            source_node,
            label=source_node,
            title=f"📄 Document: {docs[i]['filename']}",
            color="#38bdf8",
            size=18
        )
        for ent, label in entities:

            if label == "ORG":
                color = "#22c55e"   # green
            elif label == "PERSON":
                color = "#f472b6"   # pink
            elif label == "LAW":
                color = "#f59e0b"   # orange
            elif label == "GPE":
                color = "#eab308"   # yellow
            else:
                color = "#22c55e"

            G.add_node(
                ent,
                label=ent,
                title=f"""
                Entity: {ent}
                Type: {label}
                Source Document: {docs[i]['filename']}
                """,
                color=color,
                size=20
            )
            G.add_edge(source_node, ent)

        # NEW: connect entities with each other
        entity_names = [e[0] for e in entities]

        for i in range(min(len(entity_names), 5)):
            for j in range(i + 1, min(len(entity_names), 5)):
                G.add_edge(entity_names[i], entity_names[j])
    return G

def generate_interactive_graph(docs):
    if not docs: return None
    G = build_graph_from_documents(docs)

    net = Network(
        height="650px",
        width="100%",
        bgcolor="#0b0f19",
        font_color="white",
        notebook=False
    )

    net.barnes_hut(
        gravity=-8000,
        central_gravity=0.3,
        spring_length=200
    )

    for node in G.nodes():
        if "Doc:" in node:
            G.nodes[node]['color'] = "#38bdf8"
            G.nodes[node]['size'] = 30
        else:
            G.nodes[node]['color'] = "#22c55e"
            G.nodes[node]['size'] = 15

    net.from_nx(G)

    net.set_options("""
    var options = {
      "nodes": {
        "font": {
          "size": 18,
          "face": "Arial"
        }
      },
      "edges": {
        "color": {
          "color": "#1d4ed8"
        },
        "smooth": {
          "type": "dynamic"
        }
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 200,
        "navigationButtons": true,
        "zoomView": true,
        "dragView": true
      },
      "physics": {
        "barnesHut": {
          "gravitationalConstant": -20000,
          "centralGravity": 0.02,
          "springLength": 300
        },
        "stabilization": {
          "enabled": true,
          "iterations": 1000
        }
      }
    }
    """)

    output_path = os.path.join(APP_DIR, "graphs", "policy_kg.html")
    net.save_graph(output_path)
    return output_path



Writing knowledge_graph.py


In [10]:
%%writefile nlp_engine.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM
import vector_store

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
TRANSLATOR_ID = "facebook/nllb-200-distilled-600M"

# Globally cached in Streamlit
model = None
tokenizer = None
translator_model = None
translator_tokenizer = None

def init_model():
    global model, tokenizer, translator_model, translator_tokenizer
    if model is None:
        print(f"Loading {MODEL_ID} in 4-bit Quantization via BitsAndBytes...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map="auto",
            torch_dtype=torch.float16
        )

        print(f"Loading {TRANSLATOR_ID} for extreme speed translation...")
        translator_tokenizer = AutoTokenizer.from_pretrained(TRANSLATOR_ID)
        translator_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATOR_ID, device_map="auto",torch_dtype=torch.float16)
        print("Models Loaded successfully!")

# BCP-47 Code mappings for NLLB-200
LANG_CODES = {
    "English": "eng_Latn", "Hindi": "hin_Deva", "Tamil": "tam_Taml",
    "Kannada": "kan_Knda", "Telugu": "tel_Telu", "Marathi": "mar_Deva",
    "Bengali": "ben_Beng"
}

def translate_fast(text, source_lang, target_lang):
    if translator_model is None:
        init_model()
    if source_lang == target_lang: return text

    src_code = LANG_CODES.get(source_lang, "eng_Latn")
    tgt_code = LANG_CODES.get(target_lang, "eng_Latn")

    translator_tokenizer.src_lang = src_code
    inputs = translator_tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(translator_model.device)

    # Fix for AttributeError: Use convert_tokens_to_ids instead of lang_code_to_id
    tgt_token_id = translator_tokenizer.convert_tokens_to_ids(tgt_code)
    outputs = translator_model.generate(
        **inputs,
        forced_bos_token_id=tgt_token_id,
        max_length=512,
        num_beams=2
    )
    return translator_tokenizer.decode(outputs[0], skip_special_tokens=True)

def generate_english_response(prompt_text):
    if model is None: init_model()
    messages = [
        {"role": "system", "content": "You are the Public Policy Compass AI utilizing RAG. Give accurate, highly concise answers in English based closely on the context."},
        {"role": "user", "content": prompt_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=200,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def answer_policy_question(native_question, target_lang="English", simplify=False):
    # 1. Bi-directional matching: Translate their Native question into English FIRST so FAISS can search the Vector DB properly
    english_query = translate_fast(native_question, target_lang, "English")

    # 2. Search FAISS Vectors (Multiple distinct sources)
    docs = vector_store.search_documents(english_query, top_k=5)
    if not docs: docs_context = "No relevant policies found in Vector DB."
    else: docs_context = "\n\n".join([f"Snippet from {d['filename']}:\n{d['content']}" for d in docs])

    prompt = f"Context:\n{docs_context}\n\nQuestion: {english_query}\nAnswer the question using the context. Be direct."
    if simplify: prompt += " Simplify the policy language so a middle-school student can immediately understand it."

    # 3. LLM Generates English Answer
    eng_ans = generate_english_response(prompt)

    # 4. NLLB perfectly translates English Answer back to Native Language
    final_ans = translate_fast(eng_ans, "English", target_lang)
    return final_ans, docs

def summarize_document(text, target_lang="English"):
    eng_sum = generate_english_response(f"Summarize this policy into 3 highly concise bullet points:\n\n{text[:3000]}")
    return translate_fast(eng_sum, "English", target_lang)



Writing nlp_engine.py


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
%%writefile app.py
import plotly.graph_objects as go
import os
import time
import PyPDF2
from readability import ReadabilityAnalyzer
import streamlit as st
import json
from datetime import datetime, timedelta
from db import *
from auth import *
from config import *
import base64
import vector_store
import nlp_engine
import knowledge_graph
import pandas as pd
import plotly.express as px
from streamlit_option_menu import option_menu

st.set_page_config(page_title="PolicyNav Milestone_3", layout="centered")
st.markdown("""
<style>
/* Professional LLM Aesthetics */
.stApp {
    background-color: #0b0f19;
    color: #e2e8f0;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
}
div[data-testid="stSidebar"] {
    background-color: #111827 !important;
    border-right: 1px solid #1f2937;
}
div[data-testid="stChatMessage"] {
    background-color: #1e293b;
    border-radius: 12px;
    border: 1px solid #334155;
    padding: 16px;
    margin-bottom: 16px;
    box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);
}
.stButton>button {
    background-color: #ef4444;
    color: white;
    border-radius: 8px;
    font-weight: 500;
    border: 1px solid #dc2626;
    height: 42px;
    transition: all 0.2s ease;
    box-shadow: 0 1px 2px 0 rgba(0, 0, 0, 0.05);
}
.stButton>button:hover {
    background-color: #dc2626;
    border-color: #b91c1c;
    color: white;
}
div[data-baseweb="input"] > div {
    background-color: #1e293b;
    border: 1px solid #334155;
    border-radius: 8px;
    color: #f8fafc;
}
div[data-baseweb="input"] > div:hover {
    border-color: #ef4444;
}
h1, h2, h3, h4, h5, h6 {
    color: #f8fafc;
    font-weight: 600;
}
/* Title generic styling */
.project-title {
    font-weight: 800;
    font-size: 28px;
    color: #ffffff;
    padding-bottom: 16px;
    border-bottom: 1px solid #1f2937;
    margin-bottom: 24px;
}
.project-title span { color: #ef4444; }
</style>
""", unsafe_allow_html=True)


# --- NEW GENERIC HEADER ---
def render_header():
    st.markdown("<div class='project-title'><span>Public Policy</span> Navigation</div>", unsafe_allow_html=True)

init_db()
# ---------- LOAD AI MODELS ----------
@st.cache_resource
def load_and_cache_models():
    nlp_engine.init_model()
    vector_store.get_embedder()

    return True

with st.spinner("Initializing AI Core..."):
    load_and_cache_models()

# --- REUSABLE UI COMPONENTS ---
def render_feedback_ui(section_name, generated_text, unique_key):

    payload = verify_token(st.session_state.token)

    if not payload:
        return

    user_email = payload["email"]

    with st.expander(f"📝 Provide Feedback for {section_name}"):

        col1, col2 = st.columns([1,4])

        with col1:
            rating = st.radio(
                "Rating (1-5)",
                [1,2,3,4,5],
                horizontal=True,
                key=f"r_{unique_key}"
            )

        with col2:
            comments = st.text_input(
                "Comments (optional)",
                key=f"c_{unique_key}"
            )

        if st.button("Submit Feedback", key=f"fb_{unique_key}"):

            submit_feedback(
                user_email,
                section_name,
                rating,
                comments
            )

            st.toast("✅ Thank you for your feedback!")
# ------------------------------

# ---------- BACKGROUND ----------
def set_bg(image_file="99.jpg"):
    if os.path.exists(image_file):
        with open(image_file, "rb") as f:
            encoded = base64.b64encode(f.read()).decode()

        bg_style = f"""
        <style>
        .stApp {{
            background: url("data:image/jpg;base64,{encoded}") no-repeat center center fixed;
            background-size: cover;
        }}
        </style>
        """
    else:
        bg_style = """
        <style>
        .stApp {
            background: linear-gradient(135deg, #000000, #1c1c1c);
            color: white;
        }
        </style>
        """

    st.markdown(bg_style, unsafe_allow_html=True)

set_bg()

# ---------- SESSION STATE ----------
if "token" not in st.session_state:
    st.session_state.token = None

if "page" not in st.session_state:
    st.session_state.page = "Login"

if "reset_stage" not in st.session_state:
    st.session_state.reset_stage = 0

# ---------- NAVIGATION ----------
def go_to(page):
    st.session_state.page = page
    st.rerun()

# ---------- LOGIN ----------
def login_page():
    st.title("PolicyNav – Login")
    render_header()

    with st.form("login_form"):

        email = st.text_input("Email")
        password = st.text_input("Password", type="password")

        left, center, right = st.columns([1.5, 1.5, 5])

        login_btn = left.form_submit_button("Login")
        signup_btn = center.form_submit_button("Signup")
        forgot_btn = right.form_submit_button("Forgot Password")

        # ---------- LOGIN ----------
        if login_btn:

            if not email:
                st.toast("Email required", icon="⚠️")
                return

            if not password:
                st.toast("Password required", icon="⚠️")
                return

            if email == ADMIN_EMAIL and password == ADMIN_PASSWORD:
                go_to("AdminDashboard")
                return

            user = get_user_by_email(email)
            if not user:
                st.toast("Email not registered", icon="❌")
                return

            failed_attempts = user[6]
            lock_until = user[7]

            if lock_until and datetime.utcnow() < datetime.fromisoformat(lock_until):
                st.toast("Account locked. Try again later.", icon="🔒")
                return

            if not verify_text(password, user[3]):
                history = json.loads(user[8] or "[]")
                reused = any(verify_text(password, h) for h in (history[1:] if history else []))

                failed_attempts += 1

                if failed_attempts >= MAX_LOGIN_ATTEMPTS:
                    lock_time = datetime.utcnow() + timedelta(minutes=LOCK_TIME_MINUTES)
                    update_login_attempts(email, failed_attempts, lock_time.isoformat())
                    st.toast("Account locked for 5 minutes", icon="🔒")
                else:
                    update_login_attempts(email, failed_attempts)
                    if reused:
                        st.toast("Cannot login with old password; it is changed to new one.", icon="⚠️")
                    else:
                        st.toast("Invalid credentials", icon="❌")

                return

            update_login_attempts(email, 0, None)

            otp = generate_otp()
            send_otp_email(email, otp)

            st.session_state.pending_email = email
            st.session_state.otp = otp
            st.session_state.otp_time = datetime.utcnow()

            go_to("OTP")

        # ---------- SIGNUP ----------
        if signup_btn:
            go_to("Signup")

        # ---------- FORGOT PASSWORD ----------
        if forgot_btn:
            go_to("Forgot")

# ---------- OTP ----------
def otp_page():
    st.title("OTP Verification")
    render_header()

    entered = st.text_input("Enter OTP")

    if st.button("Verify OTP"):

        if "otp_time" not in st.session_state:
            st.toast("OTP expired. Please login again.", icon="⚠️")
            go_to("Login")
            return

        # Check 10-minute expiry
        if datetime.utcnow() - st.session_state.otp_time > timedelta(minutes=10):
            st.toast("OTP expired. Please login again.", icon="⏰")
            go_to("Login")
            return

        if entered == st.session_state.otp:

            # Forgot password via OTP
          if st.session_state.get("flow") == "forgot_otp":
              go_to("SetNewPassword")

           # Normal login OTP
          else:
              st.session_state.token = generate_token(st.session_state.pending_email)
              go_to("Dashboard")
        else:
            st.toast("Invalid OTP", icon="❌")

    if st.button("⬅ Back to Login"):
        go_to("Login")

# ---------- SIGNUP ----------
def signup_page():
    st.title("Signup")
    render_header()

    username = st.text_input("Username")
    email = st.text_input("Email")
    password = st.text_input("Password", type="password")
    confirm = st.text_input("Confirm Password", type="password")

    question = st.selectbox("Security Question", [
        "What is your favorite book?",
        "What is your dream job?"
    ])

    answer = st.text_input("Security Answer")

    if st.button("Create Account"):

        if not validate_email(email):
            st.toast("Invalid email format", icon="⚠️")
            return

        valid, msg = validate_password(password)
        if not valid:
            st.toast(msg, icon="⚠️")
            return

        if password != confirm:
            st.toast("Passwords do not match", icon="⚠️")
            return

        success, msg = create_user(
            username,
            email,
            hash_text(password),
            question,
            hash_text(answer)
        )

        if success:
            st.toast("Account created successfully", icon="✅")
            go_to("Login")
        else:
            st.toast(msg, icon="❌")

    if st.button("⬅ Back to Login"):
        go_to("Login")

# ---------- sidebar function ---------
def render_sidebar():
    with st.sidebar:
        st.markdown("<div style='font-weight:800;font-size:24px;color:#ffffff;margin:2px 0 20px 0;'><span style='color:#ef4444;'>Public Policy</span><br>Navigation</div>", unsafe_allow_html=True)

        page_map = {
            "Readability": 0,
            "RAG": 1,
            "Summarization": 2,
            "Graph": 3,
            "History": 4,
            "Reset": 5
        }
        current_index = page_map.get(st.session_state.page, 0) if st.session_state.page in page_map else 0

        selected = option_menu(
            menu_title=None,
            options=["Readability", "RAG Search", "Summarization", "Knowledge Graph", "History", "Reset Password", "Logout"],
            icons=["book", "search", "file-text", "diagram-3", "clock-history", "lock", "box-arrow-right"],
            default_index=current_index,
            styles={
                "container": {"padding": "0!important", "background-color": "transparent"},
                "icon": {"color": "#94a3b8", "font-size": "18px"},
                "nav-link": {"font-size": "16px", "text-align": "left", "margin": "4px 0", "--hover-color": "#1e293b", "color": "#cbd5e1", "border-radius": "8px"},
                "nav-link-selected": {"background-color": "#ef4444", "color": "#ffffff", "font-weight": "600"},
            }
        )
        # Prevent automatic redirect to Readability on first load
        if st.session_state.page == "Dashboard" and selected == "Readability":
            selected = None

        # Clear RAG chat if leaving page
        if st.session_state.page == "RAG" and selected != "RAG Search":
            st.session_state.rag_chat = []

        if st.session_state.page == "Summarization" and selected != "Summarization":
            st.session_state.doc_answer = ""
            if "summary" in st.session_state:
                st.session_state.summary = ""

        if selected == "Readability" and st.session_state.page != "Readability": go_to("Readability")
        if selected == "RAG Search" and st.session_state.page != "RAG": go_to("RAG")
        if selected == "Summarization" and st.session_state.page != "Summarization": go_to("Summarization")
        if selected == "Knowledge Graph" and st.session_state.page != "Graph": go_to("Graph")
        if selected == "History" and st.session_state.page != "History": go_to("History")
        if selected == "Reset Password" and st.session_state.page != "Reset": go_to("Reset")
        if selected == "Logout":
            st.session_state.token = None
            go_to("Login")

# ---------- DASHBOARD ----------
def dashboard_page():

    render_sidebar()
    render_header()

    payload = verify_token(st.session_state.token)

    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")
        return

    user = get_user_by_email(payload["email"])

    # MAIN DASHBOARD CONTENT
    st.title(f"Welcome, {user[1]} 👋")
    st.write("Select an option from the left menu.")

#--------- RAG search -----------------

def rag_search_tab():
    render_sidebar()
    render_header()

    payload = verify_token(st.session_state.token)

    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")

    user_email = payload["email"]

    st.markdown("<h2 style='color: #fca5a5;'>🔍 Advanced RAG Search</h2>", unsafe_allow_html=True)
    st.write("Query the Google Drive Vector Database directly. AI synthesizes answers only from policy context.")

    c1, c2 = st.columns([3, 1])
    with c1: target_lang = st.selectbox("Response Dialect (NLLB Pipeline):", ["English", "Hindi", "Tamil", "Kannada", "Telugu", "Marathi", "Bengali"])
    with c2: simplify = st.toggle("🧠 Simplify Jargon")
    st.divider()

    if "rag_chat" not in st.session_state: st.session_state.rag_chat = []

    for i, msg in enumerate(st.session_state.rag_chat):
        with st.chat_message(msg["role"]): st.markdown(msg["content"])

    if prompt := st.chat_input("Ask a policy question..."):
        st.session_state.rag_chat.append({"role": "user", "content": prompt})
        with st.chat_message("user"): st.markdown(prompt)

        with st.chat_message("assistant"):
            with st.spinner(f"Bi-Directional Vector Lookup & NLLB Extracting > Translating..."):
                t1 = time.time()
                ans, docs = nlp_engine.answer_policy_question(prompt, target_lang=target_lang, simplify=simplify)
                t2 = time.time()

                final_txt = f"**{ans}**\n\n---\n*Inference: {round(t2-t1,2)}s | 📚 Sources: {', '.join(list(set([d['filename'] for d in docs]))) if docs else 'None'}*"
                st.markdown(final_txt)

                log_activity(user_email, "RAG Search", prompt, final_txt)
                st.session_state.rag_chat.append({"role": "assistant", "content": final_txt})
                st.rerun()

    st.divider()
    render_feedback_ui("RAG Search", "General Page Feedback", "rag_global")

# --------- summarizer ----------
def summarization_tab():

    if "summary" not in st.session_state:
        st.session_state.summary = ""

    if "doc_text" not in st.session_state:
        st.session_state.doc_text = ""

    if "doc_answer" not in st.session_state:
        st.session_state.doc_answer = ""

    render_sidebar()
    render_header()

    payload = verify_token(st.session_state.token)

    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")

    user_email = payload["email"]

    st.markdown("<h2 style='color: #fca5a5;'>📝 Document Summarizer</h2>", unsafe_allow_html=True)
    st.write("Upload a PDF explicitly, or paste raw text to translate and summarize.")

    col1, col2 = st.columns([1,1])

    # -------- LEFT COLUMN (INPUT) --------
    with col1:

        uploaded_file = st.file_uploader("Upload Policy PDF or TXT", type=["pdf","txt"])

        txt = ""

        if uploaded_file is not None:

            if uploaded_file.name.endswith(".pdf"):

                try:
                    reader = PyPDF2.PdfReader(uploaded_file)
                    for page in reader.pages:
                        txt += page.extract_text() + "\n"
                except:
                    pass

            else:
                txt = uploaded_file.read().decode("utf-8")

            st.success("File Processed successfully!")

        txt_area = st.text_area("Or Paste Raw Policy Text:", value=txt, height=200)

    # -------- RIGHT COLUMN (SUMMARY BUTTON) --------
    with col2:

        lang = st.selectbox(
            "Summary Output Translation:",
            ["English","Hindi","Tamil","Kannada","Telugu","Marathi","Bengali"]
        )

        if st.button("Generate Summary", type="primary") and txt_area:

            with st.spinner("Generating Summary..."):

                txt_area = txt_area[:3000]

                summary = nlp_engine.summarize_document(
                    txt_area,
                    target_lang=lang
                )

                st.session_state.summary = summary
                st.session_state.doc_text = txt_area
                st.session_state.doc_answer = ""

                log_activity(user_email,"Summarization","Document Uploaded",summary)

    # -------- SHOW SUMMARY --------
    if st.session_state.summary:

        st.info(st.session_state.summary)

        # -------- Q&A SECTION --------
        st.divider()
        st.subheader("❓ Ask Questions About This Document")

        question = st.text_input("Ask a question from the uploaded document")

        if st.button("Get Answer") and question:

            with st.spinner("Analyzing document..."):

                context = st.session_state.doc_text[:2000]

                prompt = f"""
                Context:
                {context}

                Question: {question}

                Answer based only on the context.
                """

                answer = nlp_engine.generate_english_response(prompt)

                st.session_state.doc_answer = answer

        if st.session_state.doc_answer:
            st.success(st.session_state.doc_answer)

    st.divider()
    render_feedback_ui("Summarization","General Page Feedback","sum_global")

#---------- knowledge graph --------
def graph_tab():
    render_sidebar()
    render_header()

    payload = verify_token(st.session_state.token)

    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")

    user_email = payload["email"]

    col1, col2, col3 = st.columns(3)

    col1.metric("Documents Indexed", len(vector_store.get_all_documents()))
    col2.metric("Graph Type", "Policy Entity Network")
    col3.metric("NER Engine", "spaCy")

    st.markdown("""
    **Graph Legend**

    🟦 Document
    🟢 Organization
    🩷 Person
    🟠 Law / Policy
    🟡 Location
    """)

    st.markdown("""
    <h2 style='color:#fca5a5; font-weight:700; letter-spacing:1px;'>
    Knowledge Graph Explorer
    </h2>
    """, unsafe_allow_html=True)

    docs = vector_store.get_all_documents()
    if not docs: st.warning("No documents exist in Google Drive."); return

    if st.button("Render Interactive Topology", type="primary", use_container_width=True):
        with st.spinner("Computing Graph Layout..."):
            path = knowledge_graph.generate_interactive_graph(docs)
            if path:
                with open(path, 'r', encoding='utf-8') as f:
                    import streamlit.components.v1 as components
                    components.html(f.read(), height=650)
                log_activity(user_email, "Knowledge Graph", "Generated Graph", "Success")

    st.divider()
    render_feedback_ui("Knowledge Graph", "General Page Feedback", "kg_global")

#----------- History Tab-----------------

def overall_history_tab():
    render_header()
    st.markdown("<h2 style='color: #fca5a5;'>📜 Global Activity History</h2>", unsafe_allow_html=True)
    st.write("All your RAG searches, summarizations, and metrics from Google Drive.")

    render_sidebar()
    payload = verify_token(st.session_state.token)
    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")
    activities = get_user_activity(payload["email"])
    if not activities:
        st.info("No activity found for your account yet.")
        return

    from html import escape
    st.markdown(
        """
        <style>
        .history-table table { width: 100%; border-collapse: collapse; }
        .history-table thead th { text-align: left; padding: 10px 12px; color: #e5e7eb; background: #0b1220; border-bottom: 1px solid #334155; }
        .history-table tbody td { padding: 10px 12px; color: #e5e7eb; border-bottom: 1px solid #1f2937; }
        .history-table tbody tr:hover td { background: #0f172a; }
        .ai-output-cell { position: relative; }
        .ai-output-preview { display: inline-block; max-width: 360px; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; color: #93c5fd; }
        .hover-card { display: none; position: absolute; left: 0; top: 100%; transform: translateY(8px); background: #0b1220; border: 1px solid #334155; border-radius: 10px; padding: 12px; width: 580px; max-width: 70vw; box-shadow: 0 12px 28px rgba(0,0,0,.45); z-index: 9999; }
        .hover-card .hover-content { color: #e5e7eb; white-space: pre-wrap; font-size: 0.95rem; line-height: 1.4; }
        .ai-output-cell:hover .hover-card { display: block; }
        </style>
        """,
        unsafe_allow_html=True,
    )

    rows_html = []
    for app, user_inp, ai_out, ts in activities:
        preview = ai_out.strip() if isinstance(ai_out, str) else str(ai_out)
        trunc = (preview[:140] + "…") if len(preview) > 140 else preview
        rows_html.append(
            f"<tr>"
            f"<td>{escape(str(app))}</td>"
            f"<td>{escape(str(user_inp))}</td>"
            f"<td><div class='ai-output-cell'><span class='ai-output-preview'>{escape(trunc)}</span><div class='hover-card'><div class='hover-content'>{escape(preview)}</div></div></div></td>"
            f"<td>{escape(str(ts))}</td>"
            f"</tr>"
        )

    table_html = (
        "<div class='history-table'>"
        "<table>"
        "<thead><tr><th>App Section</th><th>Your Input</th><th>↓ AI Output</th><th>Timestamp</th></tr></thead>"
        f"<tbody>{''.join(rows_html)}</tbody>"
        "</table>"
        "</div>"
    )
    st.markdown(table_html, unsafe_allow_html=True)

# ---------- RESET ----------
def reset_page():
    st.title("🔒 Reset Password")
    render_header()

    payload = verify_token(st.session_state.token)

    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")

    user = get_user_by_email(payload["email"])

    old_password = st.text_input("Old Password", type="password")
    new_password = st.text_input("New Password", type="password")
    confirm_password = st.text_input("Confirm New Password", type="password")

    if st.button("Update Password"):

      error = False

      if not verify_text(old_password, user[3]):
        st.toast("Old password incorrect", icon="❌")
        error = True

      if new_password != confirm_password:
        st.toast("New passwords do not match", icon="⚠️")
        error = True

      history = json.loads(user[8] or "[]")

      if not error:
        for old_hash in history:
            if verify_text(new_password, old_hash):
                st.toast("Cannot reuse old password", icon="⚠️")
                error = True
                break

      if not error:
        new_hash = hash_text(new_password)

        history.insert(0, new_hash)
        history = history[:PASSWORD_HISTORY_COUNT]

        update_password(payload["email"], new_hash)
        update_password_history(payload["email"], json.dumps(history))

        st.toast("Password updated", icon="✅")
        st.session_state.token = None
        go_to("Login")

    if st.button("⬅ Back"):
        go_to("Dashboard")

# ---------- READABILITY PAGE ----------
def readability_page():
    st.title("📘 Text Readability Analyzer")
    render_header()
    render_sidebar()
    payload = verify_token(st.session_state.token)

    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")

    option = st.radio("Choose Input Type", ["Enter Text", "Upload File"])

    text = ""

    if option == "Enter Text":
        text = st.text_area("Enter text to analyze")

    else:
        uploaded_file = st.file_uploader("Upload TXT or PDF", type=["txt", "pdf"])

        if uploaded_file:

            if uploaded_file.type == "text/plain":
                text = uploaded_file.read().decode("utf-8")

            elif uploaded_file.type == "application/pdf":
                pdf_reader = PyPDF2.PdfReader(uploaded_file)
                for page in pdf_reader.pages:
                    text += page.extract_text() or ""

    if st.button("Analyze Readability"):

        if not text.strip():
            st.toast("No text found to analyze", icon="⚠️")
            return

        analyzer = ReadabilityAnalyzer(text)
        metrics = analyzer.get_all_metrics()

        st.subheader("Results")
        st.success(
        f"Overall Grade Level: {metrics['Overall Grade Level']:.2f}"
        )

        avg_grade = metrics["Overall Grade Level"]
        if avg_grade <= 6:
            level, color = 'Beginner', '#28a745'
        elif avg_grade <= 10:
            level, color = 'Intermediate', '#17a2b8'
        elif avg_grade <= 14:
            level, color = 'Advanced', '#ffc107'
        else:
            level, color = 'Expert', '#dc3545'
        st.markdown(f"<div style='font-weight:600;color:{color};'>Level: {level}</div>", unsafe_allow_html=True)

        st.subheader("Text Statistics")

        c1, c2, c3, c4, c5 = st.columns(5)

        c1.metric("Sentences", metrics["Total Sentences"])
        c2.metric("Words", metrics["Total Words"])
        c3.metric("Syllables", metrics["Total Syllables"])
        c4.metric("Complex Words", metrics["Complex Words"])
        c5.metric("Characters", metrics["Total Characters"])

        col1, col2 = st.columns(2)

        with col1:
          st.plotly_chart(create_gauge(metrics["Flesch Reading Ease"], "Flesch Reading Ease", 0, 100))
          st.caption(f"Flesch Reading Ease Value: {metrics['Flesch Reading Ease']:.2f}")

          st.plotly_chart(create_gauge(metrics["SMOG Index"], "SMOG Index", 0, 20))
          st.caption(f"SMOG Index Value: {metrics['SMOG Index']:.2f}")

          st.plotly_chart(create_gauge(metrics["Coleman-Liau"], "Coleman-Liau Index", 0, 20))
          st.caption(f"Coleman-Liau Value: {metrics['Coleman-Liau']:.2f}")

        with col2:
          st.plotly_chart(create_gauge(metrics["Flesch-Kincaid Grade"], "Flesch-Kincaid Grade", 0, 20))
          st.caption(f"Flesch-Kincaid Grade Value: {metrics['Flesch-Kincaid Grade']:.2f}")

          st.plotly_chart(create_gauge(metrics["Gunning Fog"], "Gunning Fog", 0, 20))
          st.caption(f"Gunning Fog Value: {metrics['Gunning Fog']:.2f}")

    if st.button("⬅ Back to Dashboard"):
        go_to("Dashboard")

# ------------ Gauge --------
def create_gauge(value, title, min_val, max_val):
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=value,
        title={'text': title},
        gauge={'axis': {'range': [min_val, max_val]}}
    ))
    fig.update_layout(height=250)
    return fig

# ---------- ADMIN ----------
def admin_dashboard():
    with st.sidebar:
        st.markdown("<div style='font-weight:800;font-size:24px;color:#ffffff;margin:2px 0 20px 0;'><span style='color:#ef4444;'>PolicyNav</span><br>Admin Dashboard</div>", unsafe_allow_html=True)

        selected = option_menu(
            menu_title=None,
            options=["View Users", "User Activity", "Feedback Analysis", "Refresh Data", "Logout"],
            icons=["people","clock-history","star" , "arrow-clockwise", "box-arrow-right"],
            default_index=0,
            styles={
                "container": {"padding": "0!important", "background-color": "transparent"},
                "icon": {"color": "#94a3b8", "font-size": "18px"},
                "nav-link": {"font-size": "16px", "text-align": "left", "margin": "4px 0", "--hover-color": "#1e293b", "color": "#cbd5e1", "border-radius": "8px"},
                "nav-link-selected": {"background-color": "#ef4444", "color": "#ffffff", "font-weight": "600"},
            }
        )

        if selected == "View Users":
            st.session_state.admin_view = "users"
        if selected == "Refresh Data":
            st.rerun()
        if selected == "Logout":
            go_to("Login")

    st.title("🛠 Admin Dashboard")
    render_header()

    # ---------- Feedback section -------
    if selected == "Feedback Analysis":

        st.subheader("⭐ User Feedback")

        feedback = get_all_feedback()

        if not feedback:
            st.info("No feedback submitted yet.")
            return

        df = pd.DataFrame(
            feedback,
            columns=["User Email", "Section", "Rating", "Comments", "Timestamp"]
        )

        st.dataframe(df, use_container_width=True)

        st.subheader("📊 Feedback Ratings Overview")

        chart = px.histogram(
            df,
            x="Rating",
            color="Section",
            title="Feedback Rating Distribution"
        )

        st.plotly_chart(chart, use_container_width=True)

        return
    # ---------- USER ACTIVITY ----------
    if selected == "User Activity":

        st.subheader("📜 All User Activity")

        users = get_all_users()
        emails = [u[1] for u in users]

        selected_email = st.selectbox("Select User Email", emails)

        if st.button("Load Activity Log"):

            acts = get_user_activity(selected_email)

            if not acts:
                st.info("No activity found.")
            else:
                for app, user_inp, ai_out, ts in acts:
                    st.markdown(f"**[{ts}] | Module: {app}**")
                    st.markdown(f"> **User Input:** {user_inp}")
                    st.markdown(f"> **AI Output:** {str(ai_out)[:500]}")
                    st.divider()

        return


    users = get_all_users()
    if not users:
        st.warning("No users found in database.")
        return

    st.markdown("### 👥 Manage System Users")
    st.dataframe(
        pd.DataFrame(users, columns=["Username", "Email", "Failed Attempts", "Lock Until"]),
        use_container_width=True
    )

    st.divider()

    col1, col2 = st.columns([1, 1], gap="large")

    emails = [u[1] for u in users]

    with col1:
        st.markdown("### 🔓 Account Management")
        st.info("Select an account below to view details and remove lockouts.")
        selected_email_unlock = st.selectbox("Select User Email to Unlock:", emails)
        if st.button("Unlock Account", type="primary"):
            update_login_attempts(selected_email_unlock, 0, None)
            st.success(f"{selected_email_unlock} unlocked successfully.")

    with col2:
        st.markdown("### 💬 View User Chat Activity")
        selected_email_chat = st.selectbox("Select User Email to View History:", emails)
        if st.button("Load Activity Log"):
            acts = get_user_activity(selected_email_chat)
            if not acts:
                st.info(f"No activity tracked for {selected_email_chat}.")
            else:
                with st.expander(f"Displaying {len(acts)} log(s) for {selected_email_chat}", expanded=True):
                    for i, (app, user_inp, ai_out, ts) in enumerate(acts):
                        st.markdown(f"**[{ts}] | Module: {app}**")
                        st.markdown(f"> **User Input:** {user_inp}")
                        st.markdown(f"> **AI Output:** {str(ai_out)[:500]}{'...' if len(str(ai_out)) > 500 else ''}")
                        st.divider()


# --------- forget page --------
def forgot_page():
    st.title("🔑 Forgot Password")
    render_header()

    email = st.text_input("Enter your registered email")

    method = st.radio(
        "Select Recovery Method",
        ["OTP Verification", "Security Question"]
    )

    if st.button("Continue"):

        user = get_user_by_email(email)

        if not user:
            st.toast("Email not registered", icon="❌")
            return

        st.session_state.pending_email = email

        # ⭐ Branch Logic
        if method == "OTP Verification":

            otp = generate_otp()
            send_otp_email(email, otp)

            st.session_state.otp = otp
            st.session_state.otp_time = datetime.utcnow()
            st.session_state.flow = "forgot_otp"

            go_to("OTP")

        else:
            st.session_state.flow = "forgot_security"
            go_to("SecurityQuestion")

    if st.button("⬅ Back"):
        go_to("Login")

#----- security questions ---------
def security_question_page():
    st.title("🔐 Security Verification")
    render_header()

    user = get_user_by_email(st.session_state.pending_email)

    st.write(user[4])  # security_question column

    answer = st.text_input("Your Answer")

    if st.button("Verify Answer"):

      if verify_text(answer, user[5]):

        if st.session_state.get("flow") == "forgot_security":
            go_to("SetNewPassword")
        else:
            st.toast("Unexpected flow", icon="⚠️")

      else:
        st.toast("Incorrect answer", icon="❌")

#--------- new password page ---------
def set_new_password_page():
    st.title("🔁 Set New Password")
    render_header()

    password = st.text_input("New Password", type="password")

    if st.button("Update Password"):

        valid, msg = validate_password(password)
        if not valid:
            st.toast(msg, icon="⚠️")
            return

        user = get_user_by_email(st.session_state.pending_email)

        # ✅ Load existing history
        history = json.loads(user[8] or "[]")

        # ✅ Prevent reuse of old passwords
        for old_hash in history:
            if verify_text(password, old_hash):
                st.toast("Cannot reuse old password", icon="⚠️")
                return

        new_hash = hash_text(password)

        # ✅ Update password
        update_password(st.session_state.pending_email, new_hash)

        # ✅ Update history properly
        history.insert(0, new_hash)
        history = history[:PASSWORD_HISTORY_COUNT]

        update_password_history(
            st.session_state.pending_email,
            json.dumps(history)
        )

        st.toast("Password reset successful", icon="✅")

        # Cleanup
        st.session_state.token = None
        go_to("Login")

# ---------- ROUTER ----------
page = st.session_state.page

if page == "Signup":
    signup_page()
elif page == "Dashboard":
    dashboard_page()
elif page == "Reset":
    reset_page()
elif page == "Forgot":
    forgot_page()
elif page == "OTP":
    otp_page()
elif page == "SecurityQuestion":
    security_question_page()
elif page == "SetNewPassword":
    set_new_password_page()
elif page == "AdminDashboard":
    admin_dashboard()
elif page == "Readability":
    readability_page()
elif page == "RAG":
    rag_search_tab()
elif page == "Summarization":
    summarization_tab()
elif page == "Graph":
    graph_tab()
elif page == "History":
    overall_history_tab()
else:
    login_page()


Writing app.py


In [13]:
from google.colab import userdata
import os

os.environ["JWT_SECRET_KEY"] = userdata.get("JWT_SECRET_KEY")
os.environ["EMAIL_ID"] = userdata.get("EMAIL_ID")
os.environ["EMAIL_APP_PASSWORD"] = userdata.get("EMAIL_APP_PASSWORD")
os.environ["ADMIN_EMAIL_ID"] = userdata.get("ADMIN_EMAIL_ID")
os.environ["ADMIN_PASSWORD"] = userdata.get("ADMIN_PASSWORD")

In [14]:
import os, vector_store
# Ensure the model is loaded first so we don't duplicate memory
try: _ = vector_store.get_embedder()
except: pass

docs_dir = os.path.join(APP_DIR, 'documents')
print("🔍 Scanning Google Drive for new PDF policies...")
new_chunks = vector_store.ingest_documents(docs_dir)

if new_chunks > 0:
    print(f"✅ Auto-ingested {new_chunks} new text chunks into the Vector Database!")
else:
    print("⚡ Vector Database is already up to date. Skipping ingestion. (Fast Boot)")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔍 Scanning Google Drive for new PDF policies...
Parsing new document: PMJDY_English.pdf
Parsing new document: PMJDY_2Schemes-most-visible-PM-Yojanas.pdf
Parsing new document: PMJDY_hindi.pdf
Parsing new document: PMJDY_dhan-accounts-claims-govt.pdf
Parsing new document: PMAY_Urban_PMAY Angikaar Flyer_29Aug_B.pdf
Parsing new document: PMAY_Urban_Innovative%20Technologies%20.pdf
Parsing new document: PMAY_Urban_ARH-EOI.pdf
Parsing new document: PMAY_Urban_PMAY_1_crore_Flyer.pdf
Parsing new document: JalJeevanMission_Jal-Jeevan-Samvad-November-2025.pdf
Parsing new document: JalJeevanMission_Jal-Jeevan-Samvad-December-2025.pdf
⚡ Vector Database is already up to date. Skipping ingestion. (Fast Boot)


In [15]:
# !pkill streamlit
# !pkill ngrok

In [16]:
import os, subprocess, time
from google.colab import userdata
from pyngrok import ngrok

# Explicitly ensure App Dir mapping is preserved to the shell
os.environ['APP_DIR'] = APP_DIR
ngrok_token = userdata.get('NGROK_AUTHTOKEN')

if ngrok_token:
    ngrok.set_auth_token(ngrok_token); ngrok.kill()
    process = subprocess.Popen(['streamlit', 'run', 'app.py'], env=os.environ.copy()); time.sleep(6)

    print(f"\n🔗 RAG Streamlit URL: {ngrok.connect(8501).public_url}\n")
    print("🛑 To shutdown Streamlit gracefully, press the Stop button on this Cell, or press ENTER below.")
    try: input("Press ENTER to stop...\n")
    except: pass
    finally: process.terminate(); ngrok.kill(); print("Stopped.")
else: print("❌ Ngrok Token missing from Colab secrets.")


🔗 RAG Streamlit URL: https://faf4-34-9-137-148.ngrok-free.app

🛑 To shutdown Streamlit gracefully, press the Stop button on this Cell, or press ENTER below.
Press ENTER to stop...

Stopped.
